## Summary of this Project
* external documents
> * around 1.8M characters across 7 source documents (745 pages)
* parameter configurations
> * chunk_size = 550
> * chunk_overlap = 120
> * embedding_model = "BAAI/bge-m3"
>> * embedding dimension = 1024
> * top_k = 12
> * reranker_top_k = 8
>> * reranker_model = "cross-encoder/ms-marco-MiniLM-L-6-v2"
> * llm_model = "llama-3.1-8b-instant"
> * judge_model = "openai/gpt-oss-120b"
* evalation set
> * 40 questions drafted with Claude (Anthropic), verified against source documents.
* performance metrics
> * recall: 0.97
> * answer accuracy: 0.92
> * citation validity: 0.97
>> * the proportion of generated answers in which every cited source tag refers to a chunk that was genuinely present in the retrieved context, verifying that no citation was fabricated or misattributed to a non-existent source.
> * faithfulness: 0.82
>> * the proportion of generated answers judged by an LLM-as-judge to be holistically supported by the combined text of all cited context, i.e. the answer's claims are grounded in the retrieved evidence rather than the model's own prior knowledge.
* the failures stem from the same root cause:
> * the model resolves ambiguity between multiple similar-looking candidates by relying on superficial cues (proximity, file grouping) rather than verifying the specific label/entity named in the text itself.



## 建立模組目錄

In [ ]:
## create modular root folder
import os

os.makedirs('RAG_module', exist_ok = True)
os.makedirs('RAG_module/data', exist_ok = True)

## `requirements.txt`
* 安裝列表

In [ ]:
%%writefile RAG_module/requirements.txt
llama-index
llama-index-readers-file
llama-index-embeddings-google-genai
llama-index-vector-stores-qdrant
google-genai
groq
qdrant-client
rank_bm25
sentence-transformers
jieba

Writing RAG_module/requirements.txt


In [ ]:
!pip install -q -r RAG_module/requirements.txt

## `config.py`
* 外部參數設定

In [ ]:
%%writefile RAG_module/config.py
from google.colab import userdata
import os

# external data
DOCUMENT_DIR = "RAG_module/data"

# chunking related
CHUNK_SIZE = 450
CHUNK_OVERLAP = 120

# embedding model
EMBEDDING_MODEL = "BAAI/bge-m3"
EMBEDDING_DIM = 1024
# EMBEDDING_MODEL = "gemini-embedding-001"
# EMBEDDING_DIM = 3072

# collection
COLLECTION_NAME = "demo"

# retrieval
TOP_K = 12

# reranking related
#RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
RERANKER_TOP_K = 8

# LLM
LLM_MODEL = "llama-3.1-8b-instant"
JUDGE_MODEL = "openai/gpt-oss-120b"
# LLM_MODEL = "gemini-2.0-flash-lite"
# JUDGE_MODEL = "gemini-2.0-flash-lite"

Writing RAG_module/config.py


## `api.py`

In [ ]:
%%writefile RAG_module/api.py
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

Writing RAG_module/api.py


## `test.txt`
在RAG的情境裡，就是外部文件

In [ ]:
## 解壓縮外部文件
!unzip -qj RAG_files.zip -d /content/RAG_module/data

In [ ]:
%%writefile RAG_module/data/test.txt
RAG stands for Retrieval Augmented Generation.

A RAG system retrieves relevant documents before sending context to the language model.

Qdrant is a vector database.

LlamaIndex is a framework for building RAG systems.

Gemini is a large language model developed by Google.

Writing RAG_module/data/test.txt


## `document_corrections.py`
* 讀取的外部文件，可能會有一些因編碼錯誤而在讀取時產生錯誤的文字

In [ ]:
%%writefile RAG_module/document_corrections.py

_KNOWN_OCR_ERRORS = {
    "貨幣政策工具.pdf": [
        ("店 、1,000", "500萬元、1,000"),
        ("國庫六 發 行條例", "國庫券發行條例"),
    ],
}


def apply_ocr_corrections(documents):
  # 先統計每個檔案、每筆修正，總共命中幾次
  hit_counts = {
      filename: {wrong: 0 for wrong, _ in corrections}
      for filename, corrections in _KNOWN_OCR_ERRORS.items()
  }

  for doc in documents:
    filename = doc.metadata.get("file_name")
    corrections = _KNOWN_OCR_ERRORS.get(filename)
    if not corrections:
      continue

    text = doc.get_content()
    changed = False
    for wrong, correct in corrections:
      if wrong in text:
        text = text.replace(wrong, correct)
        changed = True
        hit_counts[filename][wrong] += 1
        print(f"[ocr_correction] {filename}（page_label = "
              f"{doc.metadata.get('page_label')}）：「{wrong}」→「{correct}」")

    if changed:
      doc.set_content(text)


  # 檢查有沒有任何一筆修正，整份文件都完全沒命中
  made_corrections = True
  for filename, corrections in _KNOWN_OCR_ERRORS.items():
    for wrong, correct in corrections:
      if hit_counts.get(filename, {}).get(wrong, 0) == 0:
        made_corrections = False

  if len(_KNOWN_OCR_ERRORS) == 0:
    print(f"no corrections on the documents are needed.")

  else:
    if made_corrections:
      print(f"some known errors are corrected for the documents.")

    else:
      print(f"corrections are needed but none has been made.")

  return documents

Writing RAG_module/document_corrections.py


## `loader.py`
* 專心處理 Document
* 輸出的型態為 list of Documents

In [ ]:
%%writefile RAG_module/loader.py
from llama_index.core import (
    SimpleDirectoryReader
)

from RAG_module.config import DOCUMENT_DIR
from RAG_module.document_corrections import apply_ocr_corrections, _KNOWN_OCR_ERRORS

def load_documents():
  documents = SimpleDirectoryReader(DOCUMENT_DIR).load_data()
  documents = apply_ocr_corrections(documents)
  return documents

Writing RAG_module/loader.py


## `metadata_loader.py`
* 輸入文件層級的 metadata

In [ ]:
%%writefile RAG_module/metadata_loader.py
import csv

# 主題分類（供 metadata filter 與 BM25 領域詞彙比對使用）
CATEGORIES = [
    "支付清算",
    "貨幣政策",
    "氣候金融",
    "央行比較法制",
    "其他",
]

# 文件性質
DOC_TYPES = [
    "本行政策報告",
    "國際準則中譯本",
    "比較法選輯",
    "制度說明手冊",
    "其他",
]

_METADATA_CSV_PATH = "RAG_module/document_metadata.csv"


def load_document_metadata(csv_path=_METADATA_CSV_PATH):
  """讀取文件層級 metadata，回傳以 filename 為 key 的 lookup dict。

  回傳格式：
  {
    "貨幣政策工具.pdf": {
        "title": "貨幣政策工具",
        "category": "貨幣政策",
        "publish_date": "",
        "issuing_unit": "中央銀行",
        "doc_type": "制度說明手冊",
    },
    ...
  }
  """
  metadata_lookup = {}
  with open(csv_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
      filename = row.pop("filename")
      metadata_lookup[filename] = row
  return metadata_lookup


def get_metadata_for_file(filename, metadata_lookup):
  """取得單一檔案的 metadata；若找不到，回傳空值欄位並印出警告。

  這裡刻意不拋例外——文件蒐集階段可能會陸續加新檔案，
  找不到 metadata 時應該讓 pipeline 繼續跑，而不是整個中斷，
  但要讓使用者知道有檔案漏填 metadata。
  """
  if filename not in metadata_lookup:
    print(f"[metadata_loader] 警告：找不到 {filename} 的 metadata，"
          f"請確認 document_metadata.csv 是否有補上這筆資料。")
    return {
        "title": filename,
        "category": "其他",
        "publish_date": "",
        "issuing_unit": "",
        "doc_type": "其他",
    }
  return metadata_lookup[filename]

Writing RAG_module/metadata_loader.py


In [ ]:
## check if each document name matches its corresponding column in the metadata csv file
import unicodedata
from RAG_module.metadata_loader import load_document_metadata
from RAG_module.loader import load_documents


def check_metadata_completeness(csv_path = None):
  """檢查 CSV 裡有哪些必要欄位是空的，方便你補資料時一次看完。

  category / doc_type 是 retrieval filter 會用到的關鍵欄位，缺漏會直接
  影響檢索效果，因此獨立列出；publish_date 缺漏則僅列為提醒。
  """
  metadata_lookup = (
      load_document_metadata(csv_path) if csv_path else load_document_metadata()
  )
  missing_critical = []
  missing_date = []
  for filename, meta in metadata_lookup.items():
    if not meta.get("category") or not meta.get("doc_type"):
      missing_critical.append(filename)
    if not meta.get("publish_date"):
      missing_date.append(filename)

  if missing_critical:
    print("[test_metadata] 缺少 category 或 doc_type（建議先補齊再進 Step 3）：")
    for f in missing_critical:
      print(f"  - {f}")

  if missing_date:
    print("[test_metadata] 缺少 publish_date（可之後補，先不影響 retrieval）：")
    for f in missing_date:
      print(f"  - {f}")

  if not missing_critical and not missing_date:
    print("[test_metadata] metadata 齊全，沒有缺漏欄位。")

  return {"missing_critical": missing_critical, "missing_date": missing_date}


def verify_filenames_match(documents, csv_path=None):
  """確認 loader 讀進來的文件檔名跟 metadata csv 完全對得上。

  統一先做 NFC 正規化再比對，避免不同來源（例如 zip 解壓縮）造成的
  NFC/NFD 編碼差異誤判成「檔名不一致」。

  用法：
    from RAG_module.loader import load_documents
    from test_metadata import verify_filenames_match
    documents = load_documents()
    verify_filenames_match(documents)
  """
  metadata_lookup = (
      load_document_metadata(csv_path) if csv_path else load_document_metadata()
  )

  def _norm(name):
    return unicodedata.normalize("NFC", name) if name else name

  loader_filenames = {_norm(doc.metadata.get("file_name")) for doc in documents}
  csv_filenames = {_norm(f) for f in metadata_lookup.keys()}

  only_in_loader = loader_filenames - csv_filenames
  only_in_csv = csv_filenames - loader_filenames

  if not only_in_loader and not only_in_csv:
    print(f"[test_metadata] 檔名比對通過，{len(loader_filenames)} 份文件全部對得上 metadata。")
    return True

  if only_in_loader:
    print("[test_metadata] 這些文件有讀進來，但 CSV 裡沒有對應的 metadata：")
    for f in sorted(only_in_loader):
      print(f"  - {f}")

  if only_in_csv:
    print("[test_metadata] CSV 裡有這些檔名，但 loader 沒讀到對應文件"
          "（可能是檔案還沒放進 DOCUMENT_DIR，或檔名打錯）：")
    for f in sorted(only_in_csv):
      print(f"  - {f}")

  return False


documents = load_documents()
verify_filenames_match(documents)

[ocr_correction] 貨幣政策工具.pdf（page_label = 12）：「店 、1,000」→「500萬元、1,000」
[ocr_correction] 貨幣政策工具.pdf（page_label = 13）：「國庫六 發 行條例」→「國庫券發行條例」
some known errors are corrected for the documents.
[test_metadata] 檔名比對通過，7 份文件全部對得上 metadata。


True

## `kb_manager.py`
* 負責 knowledge base management

In [ ]:
%%writefile RAG_module/kb_manager.py
import os
import json
import hashlib

CHECKSUM_PATH = "RAG_module/checksum.json"

def compute_checksum(document_dir, embedding_model):
  hasher = hashlib.md5()
  for filename in sorted(os.listdir(document_dir)):
    filepath = os.path.join(document_dir, filename)

    if not os.path.isfile(filepath):
      continue

    with open(filepath, "rb") as f:
      hasher.update(f.read())

  hasher.update(embedding_model.encode("utf-8"))

  return hasher.hexdigest()

def load_checksum():
  if not os.path.exists(CHECKSUM_PATH):
    return None

  with open(CHECKSUM_PATH, "r") as f:
    return json.load(f).get("checksum")

def save_checksum(checksum):
  with open(CHECKSUM_PATH, "w") as f:
    json.dump({"checksum": checksum}, f)

def is_kb_outdated(document_dir, embedding_model):
  current = compute_checksum(document_dir, embedding_model)
  saved = load_checksum()
  return current != saved, current

Writing RAG_module/kb_manager.py


## `chunker.py`
* 專心處理 TextNode
* chunking 在此進行

In [ ]:
%%writefile RAG_module/chunker.py
from llama_index.core.node_parser import SentenceSplitter

def create_chunker(chunk_size, chunk_overlap):
  return SentenceSplitter(chunk_size = chunk_size,
                          chunk_overlap = chunk_overlap)


def create_nodes(documents, chunk_size, chunk_overlap):
  chunker = create_chunker(chunk_size, chunk_overlap)
  nodes = chunker.get_nodes_from_documents(documents)

  return nodes

Writing RAG_module/chunker.py


In [ ]:
## debug: 檢查「店」這個字在「貨幣政策工具.pdf」裡出現的所有位置，
## 確認要用多精確的字串才能安全替換，不誤傷其他正確內容
from RAG_module.loader import load_documents
from RAG_module.chunker import create_nodes
from RAG_module.config import CHUNK_SIZE, CHUNK_OVERLAP

documents = load_documents()
nodes = create_nodes(documents, CHUNK_SIZE, CHUNK_OVERLAP)

target_file = "貨幣政策工具.pdf"
target_nodes = [n for n in nodes if n.metadata.get("file_name") == target_file]

print("=== 「店」字出現的所有位置（前後各20字）===")
for n in target_nodes:
    content = n.get_content()
    idx = content.find("店")
    while idx != -1:
        print(f"[page_label={n.metadata.get('page_label')}]")
        print(repr(content[max(0, idx-20): idx+20]))
        print()
        idx = content.find("店", idx + 1)

print("=== 確認「國庫六發行條例」的完整上下文 ===")
for n in target_nodes:
    content = n.get_content()
    if "國庫六" in content:
        idx = content.find("國庫六")
        print(f"[page_label={n.metadata.get('page_label')}]")
        print(repr(content[max(0, idx-20): idx+20]))

[ocr_corrections correction] 貨幣政策工具.pdf（page_label = 12）：「店 、1,000」→「500萬元、1,000」
[ocr_corrections correction] 貨幣政策工具.pdf（page_label = 13）：「國庫六 發 行條例」→「國庫券發行條例」
corrections are needed but none has been made.
=== 「店」字出現的所有位置（前後各20字）===
=== 確認「國庫六發行條例」的完整上下文 ===


In [ ]:
## test: 檢視 chunker 切出來的結果，特別看表格/條列密集處切得好不好
from RAG_module.loader import load_documents
from RAG_module.chunker import create_nodes
from RAG_module.config import CHUNK_SIZE, CHUNK_OVERLAP

documents = load_documents()
nodes = create_nodes(documents, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"文件數：{len(documents)}，切出的 chunk 總數：{len(nodes)}")
print()

# 1. 每份文件切出幾個 chunk，順便看 chunk 長度分布（太短的 chunk 通常代表切壞了）
from collections import defaultdict

chunks_by_file = defaultdict(list)
for node in nodes:
    fname = node.metadata.get("file_name", "未知檔名")
    chunks_by_file[fname].append(node)

print("=== 每份文件的 chunk 數與長度統計 ===")
for fname, file_nodes in chunks_by_file.items():
    lengths = [len(n.get_content()) for n in file_nodes]
    avg_len = sum(lengths) / len(lengths)
    print(f"{fname}：{len(file_nodes)} chunks，平均長度 {avg_len:.0f} 字，"
          f"最短 {min(lengths)} 字，最長 {max(lengths)} 字")
print()

# 2. 挑幾個關鍵字，把包含這些字的 chunk 印出來，重點看表格/條列是否被切斷
#    「貨幣政策工具.pdf」裡的表格（表一~表三）跟利率數字是重點檢查對象
KEYWORDS_TO_INSPECT = ["表一", "表二", "表三", "準備率", "附買回", "重貼現"]

print("=== 包含指定關鍵字的 chunk 內容（檢查表格是否被切斷）===")
for keyword in KEYWORDS_TO_INSPECT:
    matched = [n for n in nodes if keyword in n.get_content()]
    if not matched:
        print(f"[關鍵字「{keyword}」沒有出現在任何 chunk 裡，可能被切壞或不存在]")
        continue
    print(f"--- 關鍵字「{keyword}」，共 {len(matched)} 個 chunk 命中，顯示第一個 ---")
    print(matched[0].get_content())
    print("...(chunk 結束)...")
    print()

# 3. 也印出「各國中央銀行法選輯」裡任一 chunk，檢查條號（第X條）是否完整
print("=== 各國中央銀行法選輯：隨機抽一個 chunk 看條文是否完整 ===")
for fname, file_nodes in chunks_by_file.items():
    if "中央銀行法選輯" in fname:
        sample = file_nodes[len(file_nodes) // 2]  # 抽中間那個，比較不會是封面/目錄
        print(f"[{fname}]")
        print(sample.get_content())
        print("...(chunk 結束)...")
        print()
        break

文件數：752，切出的 chunk 總數：4625

=== 每份文件的 chunk 數與長度統計 ===
一國支付系統發展之一般準則.pdf：677 chunks，平均長度 139 字，最短 2 字，最長 961 字
中華民國支付及清算系統.pdf：189 chunks，平均長度 143 字，最短 29 字，最長 242 字
各國中央銀行法選輯-2025年版-上冊.pdf：1398 chunks，平均長度 370 字，最短 0 字，最長 1391 字
各國中央銀行法選輯-2025年版-下冊.pdf：1431 chunks，平均長度 385 字，最短 0 字，最長 1124 字
因應氣候變遷策略方案.pdf：236 chunks，平均長度 150 字，最短 15 字，最長 847 字
支付與清算系統間之相互依存關係.pdf：597 chunks，平均長度 140 字，最短 1 字，最長 589 字
貨幣政策工具.pdf：97 chunks，平均長度 154 字，最短 23 字，最長 212 字

=== 包含指定關鍵字的 chunk 內容（檢查表格是否被切斷）===
[關鍵字「表一」沒有出現在任何 chunk 裡，可能被切壞或不存在]
--- 關鍵字「表二」，共 2 個 chunk 命中，顯示第一個 ---
「證券法（5728-1968）」第 1 條所定「證券」定義中，所稱
「由政府持有」等語後，接以「或由以色列銀行持有」等語。
「政府公司法（5735-1975）」附表二第18 項修正為以下內容：
「18. 依『以色列銀行法（5770-2010）』設置貨幣委員會及
理事會。」
「銀行（許可）法（5741-1981）」第 50B 條第(c)項所定「由
其決定比率」修正為「最大比率之減輕」。
...(chunk 結束)...

--- 關鍵字「表三」，共 1 個 chunk 命中，顯示第一個 ---
自民國 89 年 12 月至92 年 6 月, 中央 銀 行為刺激經濟景氣 , 調
降 各項中 央 銀 行利率 達十五次之多。有關中 央銀行利率的調整 , 可參閱 表三。
第 三節 ” 公 開 市 場操作
一、公 開 市 場操作 的意義 與目的
公開 市場操作為 中央銀 行經由 金融 市 場, 買 賣票債姜的 方式增減銀行 體
系 的準備 金,
...(chunk 結束).

In [ ]:
## test: 專項檢查「貨幣政策工具.pdf」是否有文字擷取（亂碼/缺字）問題
from RAG_module.loader import load_documents
from RAG_module.chunker import create_nodes
from RAG_module.config import CHUNK_SIZE, CHUNK_OVERLAP

documents = load_documents()
nodes = create_nodes(documents, CHUNK_SIZE, CHUNK_OVERLAP)

target_file = "貨幣政策工具.pdf"
target_nodes = [n for n in nodes if n.metadata.get("file_name") == target_file]

print(f"「{target_file}」共切出 {len(target_nodes)} 個 chunk")
print()

# 把全部 chunk 依序印出來（只有42個，全部看完也不會太長）
for i, node in enumerate(target_nodes):
    content = node.get_content()
    print(f"--- chunk {i} (長度 {len(content)} 字，page_label={node.metadata.get('page_label')}) ---")
    print(content)
    print()

「貨幣政策工具.pdf」共切出 97 個 chunk

--- chunk 0 (長度 156 字，page_label=1) ---
第四 章 ” 貨 幣政策工具
第四 章 貸 幣政策 工具
貨 幣政策 機 制為中央銀 行決策 的基本 架構 , 而 貨幣政策工具則為 中央銀
行 執行貨幣政策的手段 。發 幣政策 工具 透過對準備 發幣與金融業拆款利率的
影響 。將 影響 力傳送至其他的 經濟 金融變數 , 以 達成穩定 與發展的既定 目
慄。

--- chunk 1 (長度 106 字，page_label=1) ---
以 達成穩定 與發展的既定 目
慄。本章 以貨幣政策 工具為 主 題。依 序說明 我 國準備 金制度、貼 現窗口制
度、公開 市場操作、金 融機 構轉存款及選擇 性信用 管理 等制度的 運作與業務
操作 內 涵。

--- chunk 2 (長度 171 字，page_label=1) ---
第 一節 準備 金制度
一、準 備金制度 的意義 與目的
準備 金制度 為中央銀 行依法 要求金融機構依其負債 提存 一定比率的準備
金 , 以因應 支付需求 的制度。此處 的金融機構主要係 指具有吸收存款 與創造
貨 幣信用功能 的銀行。中 央銀 行要求 銀行提存準備 金的目的 為 :
(一 ) 銀 行本質上資產 期限長而負債 期限 短,

--- chunk 3 (長度 164 字，page_label=1) ---
透 過準備 金制度可建立 社會大眾
對 金融體系提供 充分 流動 性的信心。此 外 , 一 旦銀 行出現流動性 不 足
時 , 可 以存在中 央 銀 行的準備 金為擔保 , 向中 央銀 行申請 緊急 融通。
(二 ) 以準備 金制度中的準備 率做為 貨幣政策工具。透 過準備 率的調整。可 改
變銀行 體系 的信用 貸 放能力。

--- chunk 4 (長度 167 字，page_label=1) ---
透 過準備 率的調整。可 改
變銀行 體系 的信用 貸 放能力。進 而控制 貨幣總計數 或其他 重要金 融弦
數 , 以促進 經濟 金融的穩定 。
二 、現 行準備 金制度
我 國的準備 金制度 已行之有 年。除 於民國 83 年 11 月 為 減少拆款利率 每
月 三名週期性的波動 。將準備 金由按 旬計提 改為按月計提 以 外,

## `embedding_service.py`
* 專心處理 embedding
* 可呼叫轉換成 embedding 向量的 mapping

In [ ]:
%%writefile RAG_module/embedding_service.py
import os
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
from sentence_transformers import SentenceTransformer
# from llama_index.core import Settings
# from google import genai
# import RAG_module.api


# _client = genai.Client()
_model = None


def _get_model(embedding_model):
  global _model
  if _model is None:
    print(f"載入本地 embedding 模型：{embedding_model}（第一次執行需要下載模型檔案，請稍候）")
    _model = SentenceTransformer(embedding_model)
  return _model


def create_document_embedding(text, embedding_model):
  model = _get_model(embedding_model)
  return model.encode(text, normalize_embeddings = True).tolist()

  #result = _client.models.embed_content(model = embedding_model,
  #                                      contents = text,
  #                                      config = {'task_type': "RETRIEVAL_DOCUMENT"})

  #return result.embeddings[0].values


def create_query_embedding(text, embedding_model):
  model = _get_model(embedding_model)
  return model.encode(text, normalize_embeddings = True).tolist()

  #result = _client.models.embed_content(model = embedding_model,
  #                                      contents = text,
  #                                      config = {"task_type": "RETRIEVAL_QUERY"})


  #return result.embeddings[0].values

Writing RAG_module/embedding_service.py


In [ ]:
## check if the embedding model should be re-downloaded
import subprocess
result = subprocess.run(
    ["ls", "-lh", "/content/drive/MyDrive/hf_cache/hub/models--BAAI--bge-m3/blobs/"],
    capture_output=True, text=True
)
print(result.stdout)

total 4.3G
-rw------- 1 root root   54 Jul 18 01:27 0140ba1eac83a3c9b857d64baba91969d988624b
-rw------- 1 root root  123 Jul 18 01:27 1fba91c78a6c8e17227058ab6d4d3acb5d8630a9
-rw------- 1 root root  17M Jul 18 01:28 21106b6d7dab2952c1d496fb21d5dc9db75c28ed361a05f5020bbba27810dd08
-rw------- 1 root root  349 Jul 18 01:27 952a9b81c0bfd99800fabf352f69c7ccd46c5e43
-rw------- 1 root root 2.2G Jul 18 01:28 993b2248881724788dcab8c644a91dfd63584b6e5604ff2037cb5541e1e38e7e
-rw------- 1 root root  191 Jul 18 01:28 9bd85925f325e25246d94c4918dc02ab98f2a1b7
-rw------- 1 root root  964 Jul 18 01:28 b1879d702821e753ffe4245048eee415d54a9385
-rw------- 1 root root 2.2G Jul 18 01:28 b5e0ce3470abf5ef3831aa1bd5553b486803e83251590ab7ff35a117cf6aad38
-rw------- 1 root root 4.9M Jul 18 01:28 cfc8146abe2a0488e9e2a0c56de7952f7c11ab059eca145a0a727afce0db2865
-rw------- 1 root root  444 Jul 18 01:28 dc69ac559dcba2694012009aaa108c614541789a
-rw------- 1 root root  16K Jul 18 01:27 e5a320176edd6ee1cfb256e68ee5ac00

## `vector_store.py`

* 創建向量資料庫連線層

In [ ]:
%%writefile RAG_module/vector_store.py
from qdrant_client import (
    QdrantClient
)

def create_vector_store():
  return QdrantClient(path = "RAG_module/qdrant_db")

Writing RAG_module/vector_store.py


## `qdrant_repository.py`
* 專心處理 Qdrant
* 負責所有與 Qdrant Point/Collection 有關的操作

In [ ]:
%%writefile RAG_module/qdrant_repository.py
from qdrant_client import QdrantClient
from qdrant_client.models import (
    PointStruct,
    VectorParams,
    Distance,
    Filter,
    FieldCondition,
    MatchValue
)

# 建立一個 qdrant collection
def create_collection(client,
                      collection_name,
                      vector_size):
  if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

  client.create_collection(collection_name = collection_name,
                           vectors_config = VectorParams(size = vector_size,
                                                         distance = Distance.COSINE))


# 檢查 collection是否已經存在
def collection_exists(client, collection_name):
  return client.collection_exists(collection_name)


def create_point(idx,
                 text,
                 embedding,
                 metadata):
  payload = {"text": text}
  payload.update(metadata)
  return PointStruct(id = idx,
                     vector = embedding,
                     payload = payload)

# create a list of points from a list of nodes
def create_points(nodes, create_embedding, embedding_model):
  points = []

  for idx, node in enumerate(nodes):
    embedding = create_embedding(node.text, embedding_model)
    point = create_point(idx,
                         node.text,
                         embedding,
                         node.metadata)
    points.append(point)

  return points

# client 是程式與資料庫溝通的媒介
# 將一批 Points 寫入指定的 Collection
def upsert_points(client, collection_name, points):
  # 將資料送入資料庫
  client.upsert(collection_name = collection_name,
                points = points)

def search_points(client,
                  collection_name,
                  query_vector,
                  limit = 3,
                  filters = None):
  return client.query_points(collection_name = collection_name,
                             query = query_vector,
                             limit = limit,
                             query_filter = filters,
                             with_payload = True)

def build_metadata_filter(metadata_filters):
  if not metadata_filters:
    return None

  conditions = [FieldCondition(key = k,
                               match = MatchValue(value = v)) for k, v in metadata_filters.items()]
  return Filter(must = conditions)

Writing RAG_module/qdrant_repository.py


## `bm25_retriever.py`
* 負責 sparse search 內涵

In [ ]:
%%writefile RAG_module/bm25_retriever.py
from rank_bm25 import BM25Okapi
import jieba
import RAG_module.api

_DOMAIN_TERMS = [
    # 支付清算類
    "跨行金融資訊系統",
    "票據交換結算系統",
    "信用卡結算系統",
    "外幣結算平台",
    "中央登錄債券系統",
    "債券等殖成交系統",
    "證券劃撥結算系統",
    "票保結算系統",
    "財金公司",
    "聯卡中心",
    "集保結算所",
    "櫃買中心",
    "即時總額清算",
    "定時淨額清算",
    "混合清算",
    "款對款同步收付",
    "款券同步交割",
    "系統性風險",
    "清算風險",
    "日間透支",
    # 貨幣政策工具類
    "準備金制度",
    "貼現窗口",
    "公開市場操作",
    "選擇性信用管理",
    "短期融通",
    "擔保放款之再融通",
    "金融機構轉存款",
    "選擇性信用融通",
    "選擇性信用管制",
    "附買回協定",
    "附賣回協定",
    "中央銀行定期存單",
    "中央銀行儲蓄券",
    "不動產信用管制",
    "消費者信用管制",
    # 氣候金融類
    "氣候變遷風險",
    "有形風險",
    "轉型風險",
    "總體審慎",
    "永續投資",
    "責任投資",
    "壓力測試",
    "淨零轉型",
    "綠色金融行動方案",
    "外匯存底管理",
    # 央行比較法制類
    "中央銀行法",
    "理事會",
    "貨幣政策委員會",
    "法定資本",
    "準備銀行法",
    # 國際組織/機構類
    "國際清算銀行",
    "金融穩定委員會",
    "巴塞爾銀行監理委員會",
    "綠色金融體系網絡",
]

for _term in _DOMAIN_TERMS:
  jieba.add_word(_term)


def _tokenize(text):
  return list(jieba.cut(text.lower()))


def _matches_filters(node, metadata_filters):
  return all(node.metadata.get(k) == v for k, v in metadata_filters.items())


class BM25Retriever:
  def __init__(self, nodes):
    self.nodes = nodes
    tokenized = [_tokenize(node.text) for node in nodes]
    #tokenized = [node.text.lower().split() for node in nodes]
    # 建一個能快速查關鍵字的索引
    self.bm25 = BM25Okapi(tokenized)

  def retrieve(self, query, limit = 3, metadata_filters = None):
    tokenized_query = _tokenize(query)
    # tokenized_query = query.lower().split()
    scores = self.bm25.get_scores(tokenized_query)

    if metadata_filters:
      for i, node in enumerate(self.nodes):
        if not _matches_filters(node, metadata_filters):
          scores[i] = -1

    top_indices = sorted(range(len(scores)),
                         key = lambda i: scores[i],
                         reverse = True)[:limit]

    return [(self.nodes[i], scores[i]) for i in top_indices]

Writing RAG_module/bm25_retriever.py


In [ ]:
## test: 確認 bm25_retriever.py 裡新的 _DOMAIN_TERMS 是否讓 jieba 不再切散這些詞
import jieba
import RAG_module.bm25_retriever as bm25_retriever

print(f"已載入 {len(bm25_retriever._DOMAIN_TERMS)} 個領域詞彙\n")

broken_terms = []
for term in bm25_retriever._DOMAIN_TERMS:
  segmented = list(jieba.cut(term))
  if len(segmented) > 1:
    broken_terms.append((term, segmented))

if broken_terms:
  print(f"=== 載入後仍被切散的詞彙（共 {len(broken_terms)} 個） ===")
  for term, segmented in broken_terms:
    print(f"{term}  ->  {' / '.join(segmented)}")
else:
  print(f"全部 {len(bm25_retriever._DOMAIN_TERMS)} 個詞彙都不再被切散，載入生效。")

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
/usr/local/lib/python3.12/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG

已載入 54 個領域詞彙

全部 54 個詞彙都不再被切散，載入生效。


## `reranker.py`
* 負責 reranking
* 為了溯源引用，輸入輸出都必須更改資料型態，不能再是純文字了，這是這模組最大的更動。

In [ ]:
%%writefile RAG_module/reranker.py
from sentence_transformers import CrossEncoder

class Reranker:
  def __init__(self, model_name):
    self.model = CrossEncoder(model_name)

  def rerank(self, query, chunks, top_k):
    pairs = [(query, chunk["text"]) for chunk in chunks]
    scores = self.model.predict(pairs)
    ranked = sorted(zip(chunks, scores),
                    key = lambda x: x[1],
                    reverse = True)
    reranked = []
    for chunk, score in ranked[:top_k]:
      chunk = dict(chunk)
      chunk["rerank_score"] = float(score)
      reranked.append(chunk)
    return reranked

Writing RAG_module/reranker.py


## `prompt_builder.py`
* 建立 augmented prompt

In [ ]:
%%writefile RAG_module/prompt_builder.py
def _format_location(chunk):
  location = chunk.get("file_name") or "未知來源"
  if chunk.get("page_label"):
    location += f" 第{chunk['page_label']}頁"
  return location


def build_prompt(query, chunks):
  citation_map = {}
  context_blocks = []

  for i, chunk in enumerate(chunks, start = 1):
    tag = f"S{i}"
    citation_map[tag] = {
        "file_name": chunk.get("file_name"),
        "page_label": chunk.get("page_label"),
        "text": chunk.get("text")
    }
    context_blocks.append(f"[{tag}] ({_format_location(chunk)})\n{chunk.get('text', '')}")

  context = "\n\n".join(context_blocks)
  available_tags = ", ".join(citation_map.keys()) if citation_map else "none"

  prompt = f"""You are a RAG assistant that must cite sources.
  Answer the question only according to the context below.
  Each context block starts with a source tag like [S1], [S2].

  Rules:
  - Before answering, check whether the context actually identifies the
    specific country/institution/regulation named in the question. If the
    context does not explicitly establish that it is about that subject, say
    exactly: "I don't know." Do NOT assume unlabeled provisions belong to the
    country or institution asked about just because the surrounding structure
    or topic (e.g. "board composition", "capital requirements") matches. This
    matters especially for the comparative central bank law documents, which
    cover many countries with parallel article structures (e.g. Israel, South
    Africa, Indonesia, Turkey, Bahamas, USA) — a matching section heading does
    NOT mean it is about the country asked about.
  - When the context contains a figure (e.g. a rate, ratio, amount) that
    changed multiple times over a time series (e.g. reserve ratio or discount
    rate adjustment history), first check whether the question specifies a
    particular date or period:
      - If a specific date/period is named in the question, use the value
        that was in effect at that date/period, not the latest value.
      - If no date is specified (e.g. the question asks for the "current" or
        unqualified value), use the LAST/FINAL value in the time series
        across all context blocks, and briefly verify no later context block
        revises it further.
  - If the answer is not found in the context, say exactly: "I don't know."
  - For every factual claim in your answer, immediately add the source tag(s)
    it is based on, e.g. "...as stated in the report [S2]."
  - Only use tags that literally appear in the context below ({available_tags}).
    Never invent a tag or cite something not present in the context.
  - When there are more than one answer, use more context information to select
    and propose the best answer.

  Context: {context}
  Question: {query}
  Answer:"""

  return prompt, citation_map

Writing RAG_module/prompt_builder.py


## `gemini_client.py`
* 涉及與 LLM 模型服務的互動
* 使用 Grok 或 Gemini

In [ ]:
%%writefile RAG_module/gemini_client.py
import time
from google import genai
from groq import Groq, RateLimitError
import RAG_module.api


def setup_gemini():
  # use Google gemini
  # return genai.Client()
  return Groq()


def ask_gemini(client, prompt, llm_model, max_retries = 5):
  # 延遲避免 resource exhausted
  # time.sleep(1)

  # use Google gemini
  # response = (
  #    client.models.generate_content(model = LLM_MODEL,
  #                                   contents = prompt)
  #)
  # return response.text
  for attempt in range(max_retries):
    try:
      response = client.chat.completions.create(
          model = llm_model,
          messages = [{"role": "user", "content": prompt}]
      )
      return response.choices[0].message.content

    except RateLimitError as e:
      if attempt == max_retries - 1:
        raise

      wait_seconds = 2 ** (attempt + 1)
      print(f"{attempt + 1}/{max_retries}: [LLM] encounters rate limit, wait for {wait_seconds} seconds to retry...")
      time.sleep(wait_seconds)

Writing RAG_module/gemini_client.py


## `retrieval_pipeline.py`
* retrieval 在做以下事情時蠻容易大量重複，所以做成一個 pipeline 讓程式碼更加簡潔
> * 將 query 轉換成向量
> * 進行 dense search 或者 關鍵字詞搜尋
>> * 若要執行 metadata filtering 的話，在這裡做
> * 將上述的搜尋結果合併
> * 進行 rerank 取出前幾名

In [ ]:
%%writefile RAG_module/retrieval_pipeline.py
from RAG_module.embedding_service import create_query_embedding
from RAG_module.qdrant_repository import search_points, build_metadata_filter
from RAG_module.config import COLLECTION_NAME


def _make_chunk(text, metadata, score = None):
  return {
      "text": text,
      "file_name": metadata.get("file_name"),
      "page_label": metadata.get("page_label"),  # None if the file is .txt
      "score": score,
  }


def run_retrieval(qdrant_client, bm25_retriever, reranker, query, embedding_model, top_k, reranker_top_k, metadata_filters = None):
  query_vector = create_query_embedding(query, embedding_model)
  query_filter = build_metadata_filter(metadata_filters)

  dense_results = search_points(qdrant_client,
                                COLLECTION_NAME,
                                query_vector,
                                limit = top_k,
                                filters = query_filter).points
  bm25_results = bm25_retriever.retrieve(query, limit = top_k, metadata_filters = metadata_filters)

  dense_chunks = {p.payload["text"]: _make_chunk(p.payload["text"],
                                                 {k: v for k, v in p.payload.items() if k != "text"},
                                                 p.score) for p in dense_results}

  bm25_chunks = {node.text: _make_chunk(node.text, node.metadata, score) for node, score in bm25_results}

  # combine the results
  merged = {**dense_chunks, **bm25_chunks}  # the union in dictionary
  all_chunks = list(merged.values())

  # reranking
  reranked_chunks = reranker.rerank(query, all_chunks, reranker_top_k)

  return {
      "dense_chunks": list(dense_chunks.values()),
      "bm25_chunks": list(bm25_chunks.values()),
      "all_chunks": list(all_chunks),
      "reranked_chunks": reranked_chunks
  }

Writing RAG_module/retrieval_pipeline.py


## `citation_verifier.py`
* 檢查 LLM 回答裡引用的標籤，是不是真的存在於 citation_map 裡
> * 這時 recall 高就很重要，因為低的 recall 表示 citation_map 本身就沒有抓到關鍵訊息，再怎麼精準引用也沒有用。

In [ ]:
%%writefile RAG_module/citation_verifier.py
import re

def extract_cited_tags(answer_text):
  return set(re.findall(r"\[S\d+\]", answer_text))


def verify_citations(answer_text, citation_map):
  # check whether the labels cited truly exist in the citation map
  cited_tags = extract_cited_tags(answer_text)
  valid_keys = {f"[{tag}]" for tag in citation_map.keys()}

  valid_tags = cited_tags & valid_keys
  invalid_tags = cited_tags - valid_keys

  return {
      "cited_tags": cited_tags,
      "valid_tags": valid_tags,
      "invalid_tags": invalid_tags,
      "has_hallucinated_citation": len(invalid_tags) > 0}

Writing RAG_module/citation_verifier.py


## `faithfulness_checker.py`
* 檢查被引用的內容是否語意上甚至邏輯上支持答案或結論

In [ ]:
%%writefile RAG_module/faithfulness_checker.py
import re
import time
from groq import Groq, RateLimitError
import RAG_module.api

_client = Groq()


def _judge_faithfulness(question, answer_text, combined_source_text, judge_model, max_retries = 5):
  prompt = f"""You are a fact-checking assistant.
  You are given the ORIGINAL QUESTION, a set of SOURCE TEXT excerpts
  (separated by "---", each prefixed with a tag like [S1]), and an ANSWER
  that was generated based on these source texts.

  Determine whether the source texts, taken TOGETHER, support the answer
  as a whole. The answer must be about the same specific subject/entity/case
  as stated in the source texts, not merely about the same general topic.
  It is fine if no single excerpt alone is sufficient, as long as they
  jointly support the answer.
  Reply with only "yes" or "no".

  Original question: {question}

  Source text(s): {combined_source_text}

  Answer: {answer_text}

  Do the source texts, taken together, support this answer?"""

  for attempt in range(max_retries):
    try:
      response = _client.chat.completions.create(
          model = judge_model,
          messages = [{"role": "user", "content": prompt}]
      )
      return response.choices[0].message.content.strip().lower() == "yes"

    except RateLimitError as e:
      if attempt == max_retries - 1:
        raise
      wait_seconds = 2 ** (attempt + 1)
      print(f"{attempt + 1}/{max_retries}: [Faithfulness Checker] encounters rate limit, wait for {wait_seconds} seconds to retry...")
      time.sleep(wait_seconds)


def check_faithfulness(question, answer_text, citation_map, judge_model):
  if not citation_map:
    return {"supported": False, "reason": "no context retrieved to check against"}

  combined_source_text = "\n---\n".join(
      f"[{tag}] {info['text']}" for tag, info in citation_map.items()
  )

  supported = _judge_faithfulness(question, answer_text, combined_source_text, judge_model)

  return {"supported": supported}

Writing RAG_module/faithfulness_checker.py


## `main.py`

In [ ]:
%%writefile RAG_module/main.py
from RAG_module.config import COLLECTION_NAME, EMBEDDING_DIM, DOCUMENT_DIR
from RAG_module.config import CHUNK_SIZE, CHUNK_OVERLAP, TOP_K, RERANKER_TOP_K, LLM_MODEL, EMBEDDING_MODEL, RERANKER_MODEL, JUDGE_MODEL
from RAG_module.loader import load_documents
from RAG_module.metadata_loader import load_document_metadata, CATEGORIES, DOC_TYPES
from RAG_module.kb_manager import is_kb_outdated, save_checksum
from RAG_module.chunker import create_nodes
from RAG_module.embedding_service import create_document_embedding, create_query_embedding
from RAG_module.vector_store import create_vector_store
from RAG_module.qdrant_repository import (
    create_collection,
    collection_exists,
    create_points,
    upsert_points
)
from RAG_module.retrieval_pipeline import run_retrieval
from RAG_module.bm25_retriever import BM25Retriever
from RAG_module.reranker import Reranker
from RAG_module.prompt_builder import build_prompt
from RAG_module.gemini_client import setup_gemini, ask_gemini
from RAG_module.citation_verifier import verify_citations
from RAG_module.faithfulness_checker import check_faithfulness


def main():
  # 0: initializations
  reranker = Reranker(RERANKER_MODEL)
  gemini_client = setup_gemini()
  qdrant_client = create_vector_store()

  # 1: load the documents
  documents = load_documents()

  # 2: chunking the Documents into TextNodes with metadata extraction
  nodes = create_nodes(documents, CHUNK_SIZE, CHUNK_OVERLAP)
  metadata_lookup = load_document_metadata()
  for node in nodes:
    file_name = node.metadata.get("file_name")
    extracted = metadata_lookup.get(file_name, {})
    node.metadata.update(extracted)

  bm25_retriever = BM25Retriever(nodes)

  outdated, current_checksum = is_kb_outdated(DOCUMENT_DIR, EMBEDDING_MODEL)
  if outdated or not collection_exists(qdrant_client, COLLECTION_NAME):
    ## start building the knowledge base ##
    print("building knowledge base...")

    # 3: transform TextNodes into Points to be stored in Qdrant
    points = create_points(nodes, create_document_embedding, EMBEDDING_MODEL)

    # 4: create collection
    create_collection(qdrant_client,
                      COLLECTION_NAME,
                      EMBEDDING_DIM)

    # 5: upsert the points into the collection created
    upsert_points(qdrant_client,
                  COLLECTION_NAME,
                  points)

    # save the current checksum
    save_checksum(current_checksum)

    print("knowledge base created.")
    ## end building the knowledge base ##

  else:
    print("Using existing knowledge base.")

  print("設定檢索篩選條件（每一項都可以直接按 Enter 跳過，不套用該項）")
  print()

  metadata_filters = {}

  print("主題分類：")
  for cat in CATEGORIES:
    print(f"- {cat}")

  category_input = input("輸入主題分類關鍵字（例如「氣候」）：").strip()
  if category_input:
    matches = [c for c in CATEGORIES if category_input in c]
    if len(matches) == 1:
      metadata_filters["category"] = matches[0]
      print(f"  已套用：category = {matches[0]}")
    elif len(matches) > 1:
      print(f"  關鍵字對應到多個分類：{matches}，這項不套用篩選")
    else:
      print(f"  找不到符合的分類，這項不套用篩選")

  print()
  print("文件性質：")
  for dt in DOC_TYPES:
    print(f"- {dt}")

  doctype_input = input("輸入文件性質關鍵字（例如「政策報告」）：").strip()
  if doctype_input:
    matches = [dt for dt in DOC_TYPES if doctype_input in dt]
    if len(matches) == 1:
      metadata_filters["doc_type"] = matches[0]
      print(f"  已套用：doc_type = {matches[0]}")
    elif len(matches) > 1:
      print(f"  關鍵字對應到多個文件性質：{matches}，這項不套用篩選")
    else:
      print(f"  找不到符合的文件性質，這項不套用篩選")

  if not metadata_filters:
    metadata_filters = None
  print()

  try:

    while True:
      query = input("\nQuestion: ")

      if query.lower() == "exit":
        break

      # 6: the retrieval step
      retrieval_result = run_retrieval(qdrant_client,
                                       bm25_retriever,
                                       reranker,
                                       query,
                                       EMBEDDING_MODEL,
                                       TOP_K,
                                       RERANKER_TOP_K,
                                       metadata_filters = metadata_filters)
      reranked_chunks = retrieval_result["reranked_chunks"]

      # 7: construct the augmented prompt
      prompt, citation_map = build_prompt(query,
                                          reranked_chunks)

      # 8: obtain the LLM's response with the augmented query as input
      answer = ask_gemini(gemini_client,
                          prompt,
                          LLM_MODEL)

      print()
      print(answer)
      print()

      # check if citations are valid, meaning that they exist in the citation map
      print("本次可引用的來源：")
      for tag, info in citation_map.items():
        loc = info["file_name"] or "未知來源"
        if info["page_label"]:
          loc += f" 第{info['page_label']}頁"
        print(f"  {tag}: {loc}")
      print()

      verification = verify_citations(answer, citation_map)
      if verification["has_hallucinated_citation"]:
        print(f"⚠️ 偵測到幻覺引用：{verification['invalid_tags']}（這些標籤不存在於本次 context 中）")
      else:
        print("✅ 引用標籤檢查通過")
      print()

      # check if valid citations truly support the LLM's answer
      faithfulness = check_faithfulness(query, answer, citation_map, JUDGE_MODEL)
      if faithfulness["supported"]:
        print("✅ 引用內容語意查核通過")
      else:
        print("⚠️ 這一輪檢索到的來源，合起來似乎無法支持這個回答")
      print()

  finally:

    # close the client
    qdrant_client.close()

if __name__ == "__main__":
  main()

Writing RAG_module/main.py


In [ ]:
## 執行 main
from RAG_module.main import main

try:
  qdrant_client.close()

except:
  pass

main()

/usr/local/lib/python3.12/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
DEBUG:jieba:Loading model from cache /tmp/jieba.cache
Loading model cost 0.807 seconds.
DEBUG:jieba:Loading model cost 0.807 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Using existing knowledge base.
設定檢索篩選條件（每一項都可以直接按 Enter 跳過，不套用該項）

主題分類：
- 支付清算
- 貨幣政策
- 氣候金融
- 央行比較法制
- 其他
輸入主題分類關鍵字（例如「氣候」）：什麼是貨幣政策？
  找不到符合的分類，這項不套用篩選

文件性質：
- 本行政策報告
- 國際準則中譯本
- 比較法選輯
- 制度說明手冊
- 其他
輸入文件性質關鍵字（例如「政策報告」）：


Question: 什麼是貨幣政策？
載入本地 embedding 模型：BAAI/bge-m3（第一次執行需要下載模型檔案，請稍候）


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


貨幣政策旨在維持本國貨幣體系之健全及穩定，以確保國家經濟之穩定與所需信心。它包括使用貨幣市場操作以影響整體金融與經濟活動，故一國支付系統對貨幣政策之有效執行至為重要。 

因此，貨幣政策的目的是維持物價穩定，並支持政府之整體經濟政策，若無損於此一首要目標。 

(貨幣政策旨在維持本國貨幣體系之健全及穩定，以確保國家經濟之穩定與所需信心) [S1]
(為維持物價穩定，決定貨幣政策之原則及策略；) [S4]
(故一國支付系統對貨幣政策之有效執行至為重要。) [S3]
(只要無損於此一首要目標，貨幣政策應支持政府之整體經濟政策。) [S5]

本次可引用的來源：
  S1: 各國中央銀行法選輯-2025年版-下冊.pdf 第52頁
  S2: 各國中央銀行法選輯-2025年版-下冊.pdf 第52頁
  S3: 一國支付系統發展之一般準則.pdf 第7頁
  S4: 各國中央銀行法選輯-2025年版-上冊.pdf 第57頁
  S5: 各國中央銀行法選輯-2025年版-上冊.pdf 第14頁

✅ 引用標籤檢查通過

✅ 引用內容語意查核通過


Question: exit


# 以下進入模型表現評估

## `eval_dataset.json`
* 建立評估資料

In [ ]:
## create Evaluation Dataset
import json

dataset = [
    {"question": "Did An-Tsu take ECON 516?", "answer": "yes", "source": "ECON 516"},
    {"question": "Did An-Tsu take ECON 400?", "answer": "I don't know", "source": None},
    {"question": "Did An-Tsu take ECON 585?", "answer": "yes", "source": "ECON 585"},
    {"question": "Did An-Tsu take STAT 535?", "answer": "yes", "source": "STAT 535"},
    {"question": "Did An-Tsu take STAT 600?", "answer": "I don't know", "source": None},
    {"question": "Did An-Tsu take CSE 142?", "answer": "yes", "source": "CSE 142"},
    {"question": "What score did An-Tsu get on ECON 501?", "answer": "4.0", "source": "ECON 501"},
    {"question": "What score did An-Tsu get on ECON 499?", "answer": "I don't know", "source": None},
    {"question": "Did An-Tsu have a degree in statistics?", "answer": "yes", "source": "MASTER OF SCIENCE (STATISTICS)"},
    {"question": "What is An-Tsu highest degree?", "answer": "doctor", "source": "DOCTOR OF PHILOSOPHY (ECONOMICS)"}
]

with open("RAG_module/eval_dataset.json", "w") as f:
  json.dump(dataset, f, indent = 2)

print("eval_dataset.json created.")

eval_dataset.json created.


In [ ]:
## create evaluation dataset
import json

dataset = [
  {
    "question": "依據「因應氣候變遷策略方案」，中央銀行因應氣候變遷的政策目標有哪兩項？",
    "answer": "強化經濟金融體系因應氣候變遷風險之韌性；協助經濟體系順利轉型至永續之綠色經濟",
    "source": "強化經濟金融體系因應氣候變遷風險之韌性",
    "source_file": "因應氣候變遷策略方案.pdf",
    "expected_filters": {"category": "氣候金融"}
  },
  {
    "question": "NGFS（綠色金融體系網絡）是在哪一年、於哪個城市成立的？",
    "answer": "2017年12月，在巴黎（第一次「一個星球峰會」中）成立",
    "source": "2017 年 12 月在巴黎舉行之第一次",
    "source_file": "因應氣候變遷策略方案.pdf",
    "expected_filters": {"category": "氣候金融"}
  },
  {
    "question": "依據瑞士再保險公司資料，近10年(2012-2021年)全球極端氣候災害造成的經濟損失是多少？跟1980年代相比呢？",
    "answer": "近10年(2012-2021年)約1.93兆美元，遠高於1980年代的2,140億美元",
    "source": "1.93 兆美元",
    "source_file": "因應氣候變遷策略方案.pdf",
    "expected_filters": {"category": "氣候金融"}
  },
  {
    "question": "「綠天鵝」(green swan)這個詞指的是什麼？",
    "answer": "指氣候相關風險引發之系統性金融危機，具有災難性及不可逆性",
    "source": "綠天鵝係指氣候相關風險引發之系統性金融危機",
    "source_file": "因應氣候變遷策略方案.pdf",
    "expected_filters": None
  },
  {
    "question": "依據「貨幣政策工具」，銀行向央行申請短期融通，期限最長不能超過幾天？",
    "answer": "十天",
    "source": "期限不能超過十天",
    "source_file": "貨幣政策工具.pdf",
    "expected_filters": {"category": "貨幣政策"}
  },
  {
    "question": "我國中央銀行儲蓄券最後是什麼時候全部到期兌償完畢的？此後有沒有再發行？",
    "answer": "民國81年1月全部到期兌償完畢，此後未再發行",
    "source": "81年1月全部到期兌償完畢",
    "source_file": "貨幣政策工具.pdf",
    "expected_filters": {"category": "貨幣政策"}
  },
  {
    "question": "央行重貼現的合格票據中，工商業票據跟農業票據的重貼現期限最長分別是幾天？",
    "answer": "工商業票據最長不得超過九十天，農業票據最長不得超過一百八十天",
    "source": "農業票據最長不得超過一百八十天",
    "source_file": "貨幣政策工具.pdf",
    "expected_filters": {"category": "貨幣政策"}
  },
  {
    "question": "依據「中華民國支付及清算系統」，截至民國111年底，我國信用卡流通卡數約幾張？",
    "answer": "約5,624萬張",
    "source": "流通卡數約5,624萬張",
    "source_file": "中華民國支付及清算系統.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "在支付清算領域中，「集中交易對手」（central counterparty, CCP）的定義是什麼？",
    "answer": "擔任某些特定契約（例如在特定交易所執行之契約）所有賣方之買方及所有買方之賣方的機構",
    "source": "所有賣方之買方及所有買方之賣方的機構",
    "source_file": "支付與清算系統間之相互依存關係.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "依據這批文件，台灣2026年的信用卡循環利率上限是多少？",
    "answer": "I don't know",
    "source": None,
    "source_file": None,
    "expected_filters": None
  },

  # ===== 新增 30 題 =====
  # 支付清算（新增10題）
  {
    "question": "截至民國111年底，同資系統參加單位共幾家？包括哪些類型的機構？",
    "answer": "共86家，包括71家銀行、8家票券金融公司、中華郵政公司，以及財金公司、票交所、證交所、櫃買中心、集保結算所及聯卡中心等6家結算機構",
    "source": "同資系統參加單位共86家",
    "source_file": "中華民國支付及清算系統.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "「QR Code共通支付標準」是哪一年由財金公司協同金融機構制定的？",
    "answer": "民國106年",
    "source": "財金公司協同金融機構於民國106年制定",
    "source_file": "中華民國支付及清算系統.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "信用卡結算系統由哪個機構營運？自哪一年幾月起連結同資系統辦理清算？",
    "answer": "由聯卡中心（聯合信用卡處理中心）營運，自民國102年11月起連結同資系統辦理清算",
    "source": "信用卡結算系統由聯卡中心營運",
    "source_file": "中華民國支付及清算系統.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "民國111年信用卡結算系統處理的交易金額跟簽帳筆數約多少？",
    "answer": "交易金額約1.6兆元，筆數約13.1億筆",
    "source": "處理交易金額約1.6兆元",
    "source_file": "中華民國支付及清算系統.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "依據「一國支付系統發展之一般準則」，一般準則共有幾項？分成哪四大面向？",
    "answer": "共14項，分成銀行體系、計畫、制度性架構、基礎設施四大面向",
    "source": "一般準則總計 14 項",
    "source_file": "一國支付系統發展之一般準則.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "「一國支付系統發展之一般準則」裡的準則1，名稱是什麼？",
    "answer": "維持中央銀行居於核心地位",
    "source": "準則 1. 維持中央銀行居於核心地位",
    "source_file": "一國支付系統發展之一般準則.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "「一國支付系統發展之一般準則」裡的準則10，名稱是什麼？",
    "answer": "加強法律之確定性",
    "source": "準則 10. 加強法律之確定性",
    "source_file": "一國支付系統發展之一般準則.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "「一國支付系統發展之一般準則」裡的準則5，名稱是什麼？",
    "answer": "制訂明確之優先順序",
    "source": "準則 5. 制訂明確之優先順序",
    "source_file": "一國支付系統發展之一般準則.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "CLS是什麼機制的縮寫？主要用什麼方式清算外匯交易？",
    "answer": "CLS是Continuous Linked Settlement（持續連結清算）的縮寫，透過CLS銀行的會計帳簿，以款對款同步收付（PVP）方式進行資金撥轉",
    "source": "持續連結清算(Continuous Linked Settlement, CLS)",
    "source_file": "支付與清算系統間之相互依存關係.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "依據「支付與清算系統間之相互依存關係」，CPSS提出因應相互依存關係的三項重大挑戰是什麼？",
    "answer": "(i)採取廣泛的風險管理視野；(ii)系統、機構或服務提供者具備之風險控管機制應能與其在全球基礎設施扮演之角色相稱；(iii)在相互依存利害關係人間進行廣泛協調",
    "source": "採取廣泛的風險管理視野",
    "source_file": "支付與清算系統間之相互依存關係.pdf",
    "expected_filters": {"category": "支付清算"}
  },
  {
    "question": "SWIFT的中文全稱是什麼？",
    "answer": "環球銀行金融電信協會",
    "source": "環球銀行金融電信協會",
    "source_file": "支付與清算系統間之相互依存關係.pdf",
    "expected_filters": {"category": "支付清算"}
  },

  # 貨幣政策（新增6題）
  {
    "question": "依據「貨幣政策工具」，中央銀行的貨幣政策工具主要依序包括哪五大類制度？",
    "answer": "準備金制度、貼現窗口制度、公開市場操作、金融機構轉存款、選擇性信用管理",
    "source": "依序說明我國準備金制度、貼現窗口制度、公開市場操作",
    "source_file": "貨幣政策工具.pdf",
    "expected_filters": {"category": "貨幣政策"}
  },
  {
    "question": "支票存款、活期存款、儲蓄存款、定期存款的準備率法定上限各為多少？",
    "answer": "支票存款25%、活期存款25%、儲蓄存款15%、定期存款15%（其他各種負債25%）",
    "source": "支票存款 25%、活期存款 25%、儲蓄存款 15%、定期存款 15%",
    "source_file": "貨幣政策工具.pdf",
    "expected_filters": {"category": "貨幣政策"}
  },
  {
    "question": "中央銀行定期存單發行面額分為哪三種？",
    "answer": "500萬元、1,000萬元、與1億元三種",
    "source": "500萬元、1,000萬元、與1億元等三種",
    "source_file": "貨幣政策工具.pdf",
    "expected_filters": {"category": "貨幣政策"}
  },
  {
    "question": "銀行申請短期融通的金額，每月平均不得超過當月應提存款準備金的多少百分比？",
    "answer": "10%",
    "source": "每月平均不得超過當月應提存款準備金的10%",
    "source_file": "貨幣政策工具.pdf",
    "expected_filters": {"category": "貨幣政策"}
  },
  {
    "question": "銀行申請擔保放款之再融通，期限最長不能超過幾天？",
    "answer": "三百六十天",
    "source": "期限最長不能超過三百六十天",
    "source_file": "貨幣政策工具.pdf",
    "expected_filters": {"category": "貨幣政策"}
  },
  {
    "question": "民國87年7月後，依據哪個法規的實施，中央銀行不得再發行國庫券？",
    "answer": "「短期借款暨國庫券發行條例」",
    "source": "短期借款暨國庫券發行條例",
    "source_file": "貨幣政策工具.pdf",
    "expected_filters": {"category": "貨幣政策"}
  },

  # 氣候金融（新增5題）
  {
    "question": "依據「因應氣候變遷策略方案」，氣候變遷風險分為哪兩類？",
    "answer": "有形風險(physical risks)及轉型風險(transition risks)",
    "source": "有形風險(physical risks)及轉型風險(transition risks)",
    "source_file": "因應氣候變遷策略方案.pdf",
    "expected_filters": {"category": "氣候金融"}
  },
  {
    "question": "本行因應氣候變遷的三大核心策略是什麼？",
    "answer": "協助發展綠色永續投融資環境；積極建構本行對氣候議題之專業能力；本行營運與外匯存底管理運用納入氣候風險考量",
    "source": "協助發展綠色永續投融資環境",
    "source_file": "因應氣候變遷策略方案.pdf",
    "expected_filters": {"category": "氣候金融"}
  },
  {
    "question": "巴塞爾銀行監理委員會(BCBS)於2022年6月發布了幾項「氣候相關金融風險之有效管理及監理原則」？",
    "answer": "18項",
    "source": "發布 18 項「氣候相關金融風險之有效管理及監理原則」",
    "source_file": "因應氣候變遷策略方案.pdf",
    "expected_filters": {"category": "氣候金融"}
  },
  {
    "question": "依據本行2022年6月底的調查，40家本國銀行中有幾家已設立永續金融專責單位？",
    "answer": "29家",
    "source": "已有 29 家銀行或所屬金控公司在內部設立永續金融專責單位",
    "source_file": "因應氣候變遷策略方案.pdf",
    "expected_filters": {"category": "氣候金融"}
  },
  {
    "question": "截至2022年10月3日，NGFS有幾家央行及金融監理機構成為會員？",
    "answer": "121家",
    "source": "全球已有 121 家央行及金融監理機構成為 NGFS 成員",
    "source_file": "因應氣候變遷策略方案.pdf",
    "expected_filters": {"category": "氣候金融"}
  },

  # 央行比較法制（新增6題，含跨國混淆測試）
  {
    "question": "依據「西班牙銀行自治法」，總裁及副總裁的任期為幾年？期滿是否可連任同一職務？",
    "answer": "任期為6年，且不得續任同一職務",
    "source": "總裁及副總裁之任期為 6 年，且不得續任同一職務",
    "source_file": "各國中央銀行法選輯-2025年版-上冊.pdf",
    "expected_filters": {"category": "央行比較法制"}
  },
  {
    "question": "依據「南非準備銀行法」，總裁及副總裁的任期為幾年？",
    "answer": "任期均為5年；期滿經總統徵詢部長及理事會後，得續任命之，但任期不得逾5年",
    "source": "總裁及副總裁任期均為 5 年",
    "source_file": "各國中央銀行法選輯-2025年版-上冊.pdf",
    "expected_filters": {"category": "央行比較法制"}
  },
  {
    "question": "依據「以色列銀行法」，總裁的任期為幾年？期滿後可連任幾次？",
    "answer": "任期為5年，期滿得續派連任1次",
    "source": "總裁任期為 5 年，期滿得續派連任 1 次",
    "source_file": "各國中央銀行法選輯-2025年版-上冊.pdf",
    "expected_filters": {"category": "央行比較法制"}
  },
  {
    "question": "依據「印度尼西亞銀行法」，理事會成員的任期為幾年？",
    "answer": "5年",
    "source": "理事會成員任期 5 年",
    "source_file": "各國中央銀行法選輯-2025年版-下冊.pdf",
    "expected_filters": {"category": "央行比較法制"}
  },
  {
    "question": "依據「中華民國中央銀行法」第一條，本行經營之目標有哪四項？",
    "answer": "促進金融穩定；健全銀行業務；維護對內及對外幣值之穩定；於上列目標範圍內，協助經濟之發展",
    "source": "促進金融穩定",
    "source_file": "各國中央銀行法選輯-2025年版-上冊.pdf",
    "expected_filters": {"category": "央行比較法制"}
  },
  {
    "question": "「各國中央銀行法選輯（2025年版）」上冊跟下冊分別收錄了哪幾個國家的中央銀行法？",
    "answer": "上冊收錄西班牙、土耳其、南非、以色列；下冊收錄挪威、阿拉伯聯合大公國、印尼、巴哈馬",
    "source": "西班牙銀行自治法",
    "source_file": "各國中央銀行法選輯-2025年版-上冊.pdf",
    "expected_filters": {"category": "央行比較法制"}
  },

  # 負樣本（新增2題，故意問資料庫沒有的內容）
  {
    "question": "依據這批文件，2026年台灣中央銀行的重貼現率是多少？",
    "answer": "I don't know",
    "source": None,
    "source_file": None,
    "expected_filters": None
  },
  {
    "question": "依據這批文件，美國聯邦準備法（Federal Reserve Act）對總裁任期有何規定？",
    "answer": "I don't know",
    "source": None,
    "source_file": None,
    "expected_filters": None
  }
]

with open("RAG_module/eval_dataset.json", "w") as f:
  json.dump(dataset, f, indent = 2, ensure_ascii = False)

print(f"eval_dataset.json created with {len(dataset)} questions.")

eval_dataset.json created with 40 questions.


## `evaluator.py`
* 評估時會需要使用的各種計算
> * 包含 recall, answer quality

In [ ]:
%%writefile RAG_module/evaluator.py
import json
import re
import time
import unicodedata
import RAG_module.api
from google import genai
from groq import Groq, RateLimitError
from RAG_module.citation_verifier import verify_citations
from RAG_module.faithfulness_checker import check_faithfulness


def normalize(text):
  text = unicodedata.normalize('NFKC', text)
  return re.sub(r'\s+', '', text).lower()


def load_eval_dataset(path = "RAG_module/eval_dataset.json"):
  with open(path, "r") as f:
    return json.load(f)


def eval_retrieval(dataset, retrieve_fn, configs):
  results = []

  for item in dataset:
    if item["source"] is None:
      continue

    retrieved_texts = retrieve_fn(item["question"], configs, item.get("expected_filters"))
    hit = any(normalize(item["source"]) in normalize(text) for text in retrieved_texts)

    results.append({
        "question": item["question"],
        "source": item["source"],
        "hit": hit
    })

  recall = sum(r["hit"] for r in results) / len(results)

  return recall, results


#_client = genai.Client()
_client = Groq()

def judge_answer(question, expected, actual, judge_model, max_retries = 5):
  prompt = f"""You are an evaluation assistant.
  Judge whether the actual answer correctly answers the question.
  The correctness is based on the information provided in the expected answer.
  Note that the actual answer might provide some redundant information, which
  should not affect whether the actual answer is correct or not. Also, if both
  the expected answer and actual answer are like "I don't know", then the actual
  answer correctly answers the question. Reply with only "yes" or "no".

  Question: {question}
  Expected answer: {expected}
  Actual answer: {actual}

  Is the actual answer correct?"""

  for attempt in range(max_retries):
    try:
      response = _client.chat.completions.create(
          model = judge_model,
          messages = [{"role": "user", "content": prompt}]
      )
      return response.choices[0].message.content.strip().lower() == "yes"

    except RateLimitError as e:
      if attempt == max_retries - 1:
        raise
      wait_seconds = 2 ** (attempt + 1)
      print(f"{attempt + 1}/{max_retries}: [Judge] encounters rate limit, wait for {wait_seconds} seconds to retry...")
      time.sleep(wait_seconds)

  # with google gemini
  #response = _client.models.generate_content(
  #    model = JUDGE_MODEL,
  #    contents = prompt
  #)

  # return response.text.strip().lower() == "yes"

def eval_answer(dataset, rag_fn, configs):
  results = []

  for item in dataset:
    actual, citation_map = rag_fn(item["question"], configs, item.get("expected_filters"))

    correct = judge_answer(item["question"], item["answer"], actual, configs["judge_model"])
    citation_check = verify_citations(actual, citation_map)
    faithfulness_check = check_faithfulness(item["question"], actual, citation_map, configs["judge_model"])

    results.append({
        "question": item["question"],
        "expected": item["answer"],
        "actual": actual,
        "correct": correct,
        "has_hallucinated_citation": citation_check["has_hallucinated_citation"],
        "invalid_tags": citation_check["invalid_tags"],
        "faithfulness_supported": faithfulness_check["supported"],
    })

  n = len(results)
  scores = {
      "answer_quality": sum(r["correct"] for r in results) / n,
      "citation_validity": sum(not r["has_hallucinated_citation"] for r in results) / n,
      "faithfulness": sum(r["faithfulness_supported"] for r in results) / n,
  }

  return scores, results

Writing RAG_module/evaluator.py


## `experiment_tracker.py`
* 紀錄參數組合與對應評估指標數值

In [ ]:
%%writefile RAG_module/experiment_tracker.py
import json
import os
import pandas as pd
from datetime import datetime


TRACKER_PATH = "RAG_module/experiments.json"

def load_experiments():
  if not os.path.exists(TRACKER_PATH):
    return []
  with open(TRACKER_PATH, "r") as f:
    return json.load(f)

def save_experiment(params, metrics, notes=""):
  experiments = load_experiments()
  record = {
      "id": len(experiments) + 1,
      "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
      "params": params,
      "metrics": metrics,
      "notes": notes
  }
  experiments.append(record)
  with open(TRACKER_PATH, "w") as f:
    json.dump(experiments, f, indent=2)

  print(f"Experiment #{record['id']} saved.")

def print_experiments():
  experiments = load_experiments()
  for exp in experiments:
    print(f"\n#{exp['id']} | {exp['timestamp']}")
    print(f"  params : {exp['params']}")
    print(f"  metrics: {exp['metrics']}")
    if exp['notes']:
      print(f"  notes  : {exp['notes']}")

def show_experiments():
  experiments = load_experiments()
  if not experiments:
    print("no experiments yet")
    return

  rows = []
  for exp in experiments:
    row = {"id": exp["id"], "timestamp": exp["timestamp"]}
    row.update(exp["params"])
    row.update(exp["metrics"])
    row["notes"] = exp["notes"]
    rows.append(row)

  df = pd.DataFrame(rows)
  return df

Writing RAG_module/experiment_tracker.py


## `recall_debugger.py`
* 負責診斷與調參輔助，方便快速迭代以調高 recall

In [ ]:
%%writefile RAG_module/recall_debugger.py
from RAG_module.retrieval_pipeline import run_retrieval
from RAG_module.experiment_tracker import save_experiment
from RAG_module.chunker import create_nodes
from RAG_module.loader import load_documents
from RAG_module.bm25_retriever import BM25Retriever
from RAG_module.reranker import Reranker
from RAG_module.embedding_service import create_document_embedding
from RAG_module.qdrant_repository import create_points, create_collection, upsert_points
from RAG_module.evaluator import normalize
from RAG_module.metadata_loader import load_document_metadata
from RAG_module.config import COLLECTION_NAME, DOCUMENT_DIR


def recall_debug(qdrant_client,
                 dataset,
                 chunk_size,
                 chunk_overlap,
                 embedding_model,
                 embedding_dim,
                 reranker_model,
                 top_k,
                 reranker_top_k,
                 llm_model,
                 judge_model,
                 rebuild_kb = False,
                 notes = ""):

  # step 1: rebuild nodes with metadata
  from llama_index.core.node_parser import SentenceSplitter
  documents = load_documents()
  nodes = create_nodes(documents, chunk_size, chunk_overlap)

  metadata_lookup = load_document_metadata()
  for node in nodes:
    file_name = node.metadata.get("file_name")
    extracted = metadata_lookup.get(file_name, {})
    node.metadata.update(extracted)

  # step 2: rebuild BM25
  bm25_retriever = BM25Retriever(nodes)

  # step 3: rebuild KB if necessary
  if rebuild_kb:
    create_collection(qdrant_client, COLLECTION_NAME, embedding_dim)
    points = create_points(nodes, create_document_embedding, embedding_model)
    upsert_points(qdrant_client, COLLECTION_NAME, points)

  # step 4: perform the diagnosis for each question in the evaluation set
  reranker = Reranker(reranker_model)
  diagnosis_results = []

  for item in dataset:
    if item["source"] is None:
      continue

    source = item["source"]
    query = item["question"]
    metadata_filters = item.get("expected_filters")

    # layer 1: does the chunk exist?
    chunk_exists = any(normalize(source) in normalize(node.text) for node in nodes)

    # layer 2: are the keywords retrieved?
    result = run_retrieval(qdrant_client, bm25_retriever, reranker, query, embedding_model, top_k, reranker_top_k, metadata_filters = metadata_filters)
    dense_hit = any(normalize(source) in normalize(chunk["text"]) for chunk in result["dense_chunks"])
    bm25_hit = any(normalize(source) in normalize(chunk["text"]) for chunk in result["bm25_chunks"])

    # layer 3: are the keywords still captured after reranking?
    all_chunks = result["all_chunks"]
    rerank_hit = any(normalize(source) in normalize(chunk["text"]) for chunk in result["reranked_chunks"])

    diagnosis_results.append({
        "question": query,
        "source": source,
        "chunk_exists": chunk_exists,
        "dense_hit": dense_hit,
        "bm25_hit": bm25_hit,
        "rerank_hit": rerank_hit,
        "all_chunks": all_chunks
    })

  # step 5: print the diagnostic report
  print(f"\n{'='*60}")
  print(f"診斷參數：chunk_size = {chunk_size}, chunk_overlap = {chunk_overlap}, top_k = {top_k}, reranker_top_k = {reranker_top_k}")
  print(f"{'='*60}\n")

  recall_hits = 0
  for r in diagnosis_results:
    recall_hits += r["rerank_hit"]

    print(f"問題：{r['question']}")
    print(f"  source         : {r['source']}")
    print(f"  chunk exists   : {'✅' if r['chunk_exists'] else '❌'}")
    print(f"  dense hit      : {'✅' if r['dense_hit'] else '❌'}")
    print(f"  BM25 hit       : {'✅' if r['bm25_hit'] else '❌'}")
    print(f"  rerank hit     : {'✅' if r['rerank_hit'] else '❌'}")

    # diagnosis for implied actions
    if not r["chunk_exists"]:
      print(f"problem source: Chunking, adjust chunk_size or chunk_overlap")

    elif not r["dense_hit"] and not r["bm25_hit"]:
      print(f"problem source: Retrieval fails，consider to increase top_k or change embedding model")

    elif not r["dense_hit"]:
      print(f"problem source: keywords not in dense search but in BM25, consider to increase top_k")

    elif not r["bm25_hit"]:
      print(f"problem source: keywords not in BM25 but in dense search, consider to improve tokenization or increase top_k")

    elif not r["rerank_hit"]:
      dropped_by_reranker = [chunk for chunk in r["all_chunks"] if normalize(source) in normalize(chunk["text"])]
      print(f"problem source: Reranking, manually examine the contents for problematic chunks")
      for chunk in dropped_by_reranker:
        print(f"     ---")
        print(f"     ({chunk.get('file_name')} {chunk.get('page_label') or ''})")
        print(f"     {chunk["text"][:200]}")

      print(f"possible directions: adjust chunk_size/chunk_overlap, increase reranker_top_k, or change reranker model")

    else:
      print(f"✅ All pass")
    print()

  recall = recall_hits / len(diagnosis_results)
  print(f"{'=' * 60}")
  print(f"Recall@{reranker_top_k}: {recall:.3f}")
  print(f"{'=' * 60}\n")

  save_experiment(
      params = {"chunk_size": chunk_size,
                "chunk_overlap": chunk_overlap,
                "top_k": top_k,
                "reranker_top_k": reranker_top_k,
                "embedding_model": embedding_model,
                "reranker_model": reranker_model,
                "llm_model": llm_model,
                "judge_model": judge_model},
      metrics = {"recall": round(recall, 3)},
      notes = notes
  )

Writing RAG_module/recall_debugger.py


In [ ]:
## run the recall debugger
# llm_model and judge_model do not matter
from RAG_module.recall_debugger import recall_debug
from RAG_module.vector_store import create_vector_store
from RAG_module.evaluator import load_eval_dataset

recall_configs = {
    'chunk_size': 550,
    'chunk_overlap': 150,
    'embedding_model': "BAAI/bge-m3",
    'embedding_dim': 1024,
    'top_k': 12,
    'reranker_model': "BAAI/bge-reranker-v2-m3",
    'reranker_top_k': 8,
    'llm_model': "llama-3.1-8b-instant",
    'judge_model': "openai/gpt-oss-120b"
}

try:
  qdrant_client.close()

except:
  pass

qdrant_client = create_vector_store()
dataset = load_eval_dataset()

recall_debug(
    qdrant_client = qdrant_client,
    dataset = dataset,
    chunk_size = recall_configs['chunk_size'],
    chunk_overlap = recall_configs['chunk_overlap'],
    embedding_model = recall_configs['embedding_model'],
    embedding_dim = recall_configs['embedding_dim'],
    reranker_model = recall_configs['reranker_model'],
    top_k = recall_configs['top_k'],
    reranker_top_k = recall_configs['reranker_top_k'],
    llm_model = recall_configs['llm_model'],
    judge_model = recall_configs['judge_model'],
    rebuild_kb = True
)

[ocr_correction] 貨幣政策工具.pdf（page_label = 12）：「店 、1,000」→「500萬元、1,000」
[ocr_correction] 貨幣政策工具.pdf（page_label = 13）：「國庫六 發 行條例」→「國庫券發行條例」
some known errors are corrected for the documents.
載入本地 embedding 模型：BAAI/bge-m3（第一次執行需要下載模型檔案，請稍候）


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:01<?, ?it/s]


診斷參數：chunk_size = 550, chunk_overlap = 150, top_k = 12, reranker_top_k = 8

問題：依據「因應氣候變遷策略方案」，中央銀行因應氣候變遷的政策目標有哪兩項？
  source         : 強化經濟金融體系因應氣候變遷風險之韌性
  chunk exists   : ✅
  dense hit      : ✅
  BM25 hit       : ✅
  rerank hit     : ✅
✅ All pass

問題：NGFS（綠色金融體系網絡）是在哪一年、於哪個城市成立的？
  source         : 2017 年 12 月在巴黎舉行之第一次
  chunk exists   : ✅
  dense hit      : ✅
  BM25 hit       : ✅
  rerank hit     : ✅
✅ All pass

問題：依據瑞士再保險公司資料，近10年(2012-2021年)全球極端氣候災害造成的經濟損失是多少？跟1980年代相比呢？
  source         : 1.93 兆美元
  chunk exists   : ✅
  dense hit      : ✅
  BM25 hit       : ✅
  rerank hit     : ✅
✅ All pass

問題：「綠天鵝」(green swan)這個詞指的是什麼？
  source         : 綠天鵝係指氣候相關風險引發之系統性金融危機
  chunk exists   : ✅
  dense hit      : ✅
  BM25 hit       : ❌
  rerank hit     : ✅
problem source: keywords not in BM25 but in dense search, consider to improve tokenization or increase top_k

問題：依據「貨幣政策工具」，銀行向央行申請短期融通，期限最長不能超過幾天？
  source         : 期限不能超過十天
  chunk exists   : ❌
  dense hit      : ❌
  BM25 hit       : ❌


In [ ]:
## print the experiments of recall debugging
from RAG_module.experiment_tracker import show_experiments
show_experiments()

,id,timestamp,chunk_size,chunk_overlap,top_k,reranker_top_k,embedding_model,reranker_model,llm_model,judge_model,recall,notes
0,1,2026-07-22 04:02,550,150,12,8,BAAI/bge-m3,BAAI/bge-reranker-v2-m3,llama-3.1-8b-instant,qwen/qwen3.6-27b,0.97,


## `answer_quality_debugger.py`
* 負責診斷與調參輔助，方便快速迭代以調高 answer quality
* 這邊雖然會計算其他指標，但是調參主角是 answer quality

In [ ]:
%%writefile RAG_module/answer_quality_debugger.py
from RAG_module.retrieval_pipeline import run_retrieval
from RAG_module.evaluator import normalize, judge_answer
from RAG_module.reranker import Reranker
from RAG_module.prompt_builder import build_prompt
from RAG_module.gemini_client import ask_gemini
from RAG_module.experiment_tracker import save_experiment
from RAG_module.loader import load_documents
from RAG_module.chunker import create_nodes
from RAG_module.metadata_loader import load_document_metadata
from RAG_module.bm25_retriever import BM25Retriever
from RAG_module.citation_verifier import verify_citations
from RAG_module.faithfulness_checker import check_faithfulness

def answer_quality_debug(qdrant_client,
                         gemini_client,
                         dataset,
                         chunk_size,
                         chunk_overlap,
                         embedding_model,
                         embedding_dim,
                         reranker_model,
                         top_k,
                         reranker_top_k,
                         llm_model,
                         judge_model,
                         notes = ""):

  # step 1: initialize reranker
  from llama_index.core.node_parser import SentenceSplitter
  documents = load_documents()
  nodes = create_nodes(documents, chunk_size, chunk_overlap)

  metadata_lookup = load_document_metadata()
  for node in nodes:
    file_name = node.metadata.get("file_name")
    extracted = metadata_lookup.get(file_name, {})
    node.metadata.update(extracted)

  bm25_retriever = BM25Retriever(nodes)
  reranker = Reranker(reranker_model)

  # step 2: perform the diagnosis for each question in the evaluation set
  diagnosis_results = []

  for item in dataset:
    query = item["question"]
    expected = item["answer"]
    source = item["source"]
    metadata_filters = item.get("expected_filters")

    # run retrieval
    result = run_retrieval(qdrant_client, bm25_retriever, reranker, query, embedding_model, top_k, reranker_top_k, metadata_filters = metadata_filters)

    # recall 有沒有 hit（跟 recall_debugger 邏輯一致）
    if source is None:
      recall_hit = None  # 這題不適用 recall 判斷

    else:
      recall_hit = any(normalize(source) in normalize(chunk["text"]) for chunk in result["all_chunks"])

    # rerank
    reranked_chunks = result["reranked_chunks"]

    # LLM 回答
    prompt, citation_map = build_prompt(query, reranked_chunks)
    actual = ask_gemini(gemini_client, prompt, llm_model)

    # judge
    correct = judge_answer(query, expected, actual, judge_model)

    # citation verification and faithfulness checking
    citation_check = verify_citations(actual, citation_map)
    faithfulness_check = check_faithfulness(query, actual, citation_map, judge_model)

    diagnosis_results.append({
        "question": query,
        "expected": expected,
        "actual": actual,
        "correct": correct,
        "recall_hit": recall_hit,
        "has_hallucinated_citation": citation_check["has_hallucinated_citation"],
        "invalid_tags": citation_check["invalid_tags"],
        "faithfulness_supported": faithfulness_check["supported"],
    })

  # step 3: print the diagnostic report and save
  print(f"\n{'='*60}")
  print(f"Answer Quality Diagnosis | top_k = {top_k}, reranker_top_k = {reranker_top_k}")
  print(f"{'='*60}\n")

  correct_count = 0
  for r in diagnosis_results:
    correct_count += r["correct"]

    print(f"Question: {r['question']}")
    print(f"  expected     : {r['expected']}")
    print(f"  actual       : {r['actual'].strip()}")
    print(f"  correct      : {'✅' if r['correct'] else '❌'}")
    print(f"  citation OK  : {'❌ 幻覺引用 ' + str(r['invalid_tags']) if r['has_hallucinated_citation'] else '✅'}")
    print(f"  faithfulness : {'✅' if r['faithfulness_supported'] else '❌'}")

    if not r["correct"]:
      if r["recall_hit"] is False:
        print(f"problem source: Retrieval, recall does not hit, come back to Phase 1")

      elif r["recall_hit"] is None:
        print(f"the expected answer is 'I don't know', check if the actual answer is semantically equivalent to the expected one")

      else:
        print(f"Recall hits but the answer is wrong, see below")
        print(f"if the actual answer is in fact correct: Judge, consider more powerful judge models")
        print(f"if the actual answer is truly incorrect: Prompt, consider to change the prompt_builder")
    else:
      print(f"✅ All pass")

    print()

  score = correct_count / len(diagnosis_results)
  citation_validity = sum(not r["has_hallucinated_citation"] for r in diagnosis_results) / len(diagnosis_results)
  faithfulness_score = sum(r["faithfulness_supported"] for r in diagnosis_results) / len(diagnosis_results)

  print(f"{'=' * 60}")
  print(f"Answer Quality: {score:.3f}")
  print(f"{'=' * 60}\n")

  # record the results
  save_experiment(
      params = {"chunk_size": chunk_size,
                "chunk_overlap": chunk_overlap,
                "top_k": top_k,
                "reranker_top_k": reranker_top_k,
                "embedding_model": embedding_model,
                "reranker_model": reranker_model,
                "llm_model": llm_model,
                "judge_model": judge_model},
      metrics = {"answer_quality": round(score, 3),
                "citation_validity": round(citation_validity, 3),
                "faithfulness": round(faithfulness_score, 3)},
      notes = notes
  )

Writing RAG_module/answer_quality_debugger.py


In [ ]:
## run the answer-quality debugger
from RAG_module.vector_store import create_vector_store
from RAG_module.evaluator import load_eval_dataset
from RAG_module.gemini_client import setup_gemini
from RAG_module.answer_quality_debugger import answer_quality_debug

answer_quality_configs = {
    'chunk_size': 550,
    'chunk_overlap': 150,
    'embedding_model': "BAAI/bge-m3",
    'embedding_dim': 1024,
    'top_k': 12,
    'reranker_model': "BAAI/bge-reranker-v2-m3",
    'reranker_top_k': 8,
    'llm_model': "llama-3.1-8b-instant",
    'judge_model': "openai/gpt-oss-120b"
}

try:
  qdrant_client.close()

except:
  pass

qdrant_client = create_vector_store()
dataset = load_eval_dataset()
gemini_client = setup_gemini()

answer_quality_debug(
    qdrant_client = qdrant_client,
    gemini_client = gemini_client,
    dataset = dataset,
    chunk_size = answer_quality_configs['chunk_size'],
    chunk_overlap = answer_quality_configs['chunk_overlap'],
    embedding_model = answer_quality_configs['embedding_model'],
    embedding_dim = answer_quality_configs['embedding_dim'],
    reranker_model = answer_quality_configs['reranker_model'],
    top_k = answer_quality_configs['top_k'],
    reranker_top_k = answer_quality_configs['reranker_top_k'],
    llm_model = answer_quality_configs['llm_model'],
    judge_model = answer_quality_configs['judge_model']
)

[ocr_correction] 貨幣政策工具.pdf（page_label = 12）：「店 、1,000」→「500萬元、1,000」
[ocr_correction] 貨幣政策工具.pdf（page_label = 13）：「國庫六 發 行條例」→「國庫券發行條例」
some known errors are corrected for the documents.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

1/5: [LLM] encounters rate limit, wait for 2 seconds to retry...
1/5: [LLM] encounters rate limit, wait for 2 seconds to retry...

Answer Quality Diagnosis | top_k = 12, reranker_top_k = 8

Question: 依據「因應氣候變遷策略方案」，中央銀行因應氣候變遷的政策目標有哪兩項？
  expected     : 強化經濟金融體系因應氣候變遷風險之韌性；協助經濟體系順利轉型至永續之綠色經濟
  actual       : 根據上述文獻，我們可以得出以下結論：

中央銀行因應氣候變遷的政策目標分別為：

1. 「強化經濟金融體系因應氣候變遷風險之韌性」
2. 「協助經濟體系順利轉型至永續之綠色經濟」

這兩個政策目標根據文獻[S1]、[S3]、[S4]、[S5]和[S8]所提到的「因應氣候變遷策略方案」得出的結論[source]。
  correct      : ✅
  citation OK  : ✅
  faithfulness : ✅
✅ All pass

Question: NGFS（綠色金融體系網絡）是在哪一年、於哪個城市成立的？
  expected     : 2017年12月，在巴黎（第一次「一個星球峰會」中）成立
  actual       : NGFS（綠色金融體系網絡）是在 2017 年 12 月，在巴黎成立的，源自於該年的第一個 " 一個星球峰會" (One Planet Summit)。 [S1] (因應氣候變遷策略方案.pdf 第20頁) [S29]
  correct      : ✅
  citation OK  : ❌ 幻覺引用 {'[S29]'}
  faithfulness : ✅
✅ All pass

Question: 依據瑞士再保險公司資料，近10年(2012-2021年)全球極端氣候災害造成的經濟損失是多少？跟1980年代相比呢？
  expected     : 近10年(2012-2021年)約1.93兆美元，遠高於1980年代的2,140億美元
  actual       : 根據資料 [S1]和資料 [S2]，近 1

In [ ]:
## print the experiments of answer-quality debugging
from RAG_module.experiment_tracker import show_experiments
show_experiments()

,id,timestamp,chunk_size,chunk_overlap,top_k,reranker_top_k,embedding_model,reranker_model,llm_model,judge_model,recall,notes,answer_quality,citation_validity,faithfulness
0,1,2026-07-22 04:02,550,150,12,8,BAAI/bge-m3,BAAI/bge-reranker-v2-m3,llama-3.1-8b-instant,qwen/qwen3.6-27b,0.97,,NaN,NaN,NaN
1,2,2026-07-22 04:24,550,150,12,8,BAAI/bge-m3,BAAI/bge-reranker-v2-m3,llama-3.1-8b-instant,qwen/qwen3.6-27b,NaN,,0.00,1.00,0.00
2,3,2026-07-22 04:47,550,150,12,8,BAAI/bge-m3,BAAI/bge-reranker-v2-m3,llama-3.1-8b-instant,openai/gpt-oss-120b,NaN,,0.93,0.97,0.82


## 評估主程式
* `main.py` 是互動式的設計；但 RAG 模型表現評估是需要自動跑完所有評估資料問題，與 `main.py` 不相容。
* 可視為經歷過調參的最終測驗，測驗結果做為是否能拿到證照的依據。
> * 診斷時經歷各種寫考古題與檢討的過程，運行評估主程式就像是參加正式測驗。

In [ ]:
import RAG_module.api
from RAG_module.retrieval_pipeline import run_retrieval
from RAG_module.evaluator import load_eval_dataset, eval_retrieval, eval_answer
from RAG_module.bm25_retriever import BM25Retriever
from RAG_module.reranker import Reranker
from RAG_module.vector_store import create_vector_store
from RAG_module.loader import load_documents
from RAG_module.chunker import create_nodes
from RAG_module.metadata_loader import load_document_metadata
from RAG_module.experiment_tracker import save_experiment, print_experiments
from RAG_module.prompt_builder import build_prompt
from RAG_module.gemini_client import setup_gemini, ask_gemini
from RAG_module.config import COLLECTION_NAME


evaluation_configs = {
    'chunk_size': 256,
    'chunk_overlap': 50,
    'embedding_model': "BAAI/bge-m3",
    'embedding_dim': 1024,
    'top_k': 8,
    'reranker_model': "cross-encoder/ms-marco-MiniLM-L-6-v2",
    'reranker_top_k': 8,
    'llm_model': "llama-3.1-8b-instant",
    'judge_model': "llama-3.3-70b-versatile"
}

# 0: close any possibly existent Qdrant client
try:
  qdrant_client.close()

except:
  pass


# 1: initializations
qdrant_client = create_vector_store()
documents = load_documents()
nodes = create_nodes(documents, evaluation_configs['chunk_size'], evaluation_configs['chunk_overlap'])

metadata_lookup = load_document_metadata()
for node in nodes:
  file_name = node.metadata.get("file_name")
  extracted = metadata_lookup.get(file_name, {})
  node.metadata.update(extracted)

bm25_retriever = BM25Retriever(nodes)
reranker = Reranker(evaluation_configs['reranker_model'])
gemini_client = setup_gemini()


# 2: define retrieve_fn, retrieve_fn_no_rerank and reg_fn
# retrieve_fn: extract the texts from the output of reranking
def retrieve_fn(query, configs, metadata_filters = None):
  result = run_retrieval(qdrant_client,
                         bm25_retriever,
                         reranker,
                         query,
                         configs['embedding_model'],
                         configs['top_k'],
                         configs['reranker_top_k'],
                         metadata_filters = metadata_filters)
  return [chunk["text"] for chunk in result["reranked_chunks"]]


def retrieve_fn_no_rerank(query, configs, metadata_filters = None):
  query_vector = create_query_embedding(query, configs['embedding_model'])
  query_filter = build_metadata_filter(metadata_filters)
  dense_results = search_points(qdrant_client,
                                COLLECTION_NAME,
                                query_vector,
                                limit = configs['top_k'],
                                filters = query_filter).points
  bm25_results = bm25_retriever.retrieve(query, limit = configs['top_k'], metadata_filters = metadata_filters)

  dense_chunks = {p.payload["text"]: _make_chunk(p.payload["text"],
                                                 {k: v for k, v in p.payload.items() if k != "text"},
                                                 p.score) for p in dense_results}
  bm25_chunks = {node.text: _make_chunk(node.text, node.metadata, score) for node, score in bm25_results}

  merged = {**dense_chunks, **bm25_chunks}  # the union in dictionary
  all_chunks = list(merged.values())

  return [chunk["text"] for chunk in all_chunks]


def rag_fn_with_citation(query, configs, metadata_filters = None):
  result = run_retrieval(qdrant_client,
                         bm25_retriever,
                         reranker,
                         query,
                         configs['embedding_model'],
                         configs['top_k'],
                         configs['reranker_top_k'],
                         metadata_filters = metadata_filters)
  prompt, citation_map = build_prompt(query, result["reranked_chunks"])
  answer = ask_gemini(gemini_client, prompt, configs['llm_model'])
  return answer, citation_map


# 3: evaluation stage 1: recall
with_rerank = True
dataset = load_eval_dataset()
recall, results = eval_retrieval(dataset, retrieve_fn, evaluation_configs)

if with_rerank:
  print(f"Recall@{evaluation_configs['reranker_top_k']}: {recall:.2f}\n")

else:
  print(f"Recall@{evaluation_configs['top_k']}: {recall:.2f}\n")


for r in results:
  status = "✅" if r["hit"] else "❌"
  print(f"{status} {r['question']}")


# 4: evaluation stage 2: answer quality and other secondary metrics
answer_scores, answer_results = eval_answer(dataset, rag_fn_with_citation, evaluation_configs)

aq_score = answer_scores["answer_quality"]
citation_validity = answer_scores["citation_validity"]
faithfulness_score = answer_scores["faithfulness"]

print(f"Answer Quality: {aq_score:.2f}")
print(f"Citation Validity: {citation_validity:.2f}")
print(f"Faithfulness: {faithfulness_score:.2f}\n")

for r in answer_results:
  status = "✅" if r["correct"] else "❌"
  print(f"{status} {r['question']}")
  print(f"   expected: {r['expected']}")
  print(f"   actual  : {r['actual'].strip()}")
  if r["has_hallucinated_citation"]:
    print(f"   ⚠️ invalid tags: {r['invalid_tags']}")
  if not r["faithfulness_supported"]:
    print(f"   ⚠️ 這一輪檢索到的來源，合起來似乎無法支持這個回答")
  print()
  print()

# 5: record the experiment result
params = {
    "chunk_size": evaluation_configs['chunk_size'],
    "chunk_overlap": evaluation_configs['chunk_overlap'],
    "top_k": evaluation_configs['top_k'],
    "reranker_top_k": evaluation_configs['reranker_top_k'],
    "embedding_model": evaluation_configs['embedding_model'],
    "reranker_model": evaluation_configs['reranker_model'],
    "llm_model": evaluation_configs['llm_model'],
    "judge_model": evaluation_configs['judge_model']
}

metrics = {
    "recall": round(recall, 2),
    "answer_quality": round(aq_score, 2),
    "citation_validity": round(citation_validity, 2),
    "faithfulness": round(faithfulness_score, 2)
}

save_experiment(params, metrics, notes = "")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Recall@8: 0.78

✅ 依據「因應氣候變遷策略方案」，中央銀行因應氣候變遷的政策目標有哪兩項？
✅ NGFS（綠色金融體系網絡）是在哪一年、於哪個城市成立的？
✅ 依據瑞士再保險公司資料，近10年(2012-2021年)全球極端氣候災害造成的經濟損失是多少？跟1980年代相比呢？
✅ 「綠天鵝」(green swan)這個詞指的是什麼？
❌ 依據「貨幣政策工具」，銀行向央行申請短期融通，期限最長不能超過幾天？
✅ 我國中央銀行儲蓄券最後是什麼時候全部到期兌償完畢的？此後有沒有再發行？
✅ 央行重貼現的合格票據中，工商業票據跟農業票據的重貼現期限最長分別是幾天？
❌ 依據「中華民國支付及清算系統」，截至民國111年底，我國信用卡流通卡數約幾張？
✅ 在支付清算領域中，「集中交易對手」（central counterparty, CCP）的定義是什麼？
Answer Quality: 0.80
Citation Validity: 1.00
Faithfulness: 0.80

✅ 依據「因應氣候變遷策略方案」，中央銀行因應氣候變遷的政策目標有哪兩項？
   expected: 強化經濟金融體系因應氣候變遷風險之韌性；協助經濟體系順利轉型至永續之綠色經濟
   actual  : 根據[S2](因應氣候變遷策略方案.pdf 第5頁)的描述，中央銀行將「強化經濟金融體系因應氣候變遷風險之韌性」及「協助經濟體系順利轉型至永續之綠色經濟」訂為因應氣候變遷之政策目標。

因此，中央銀行因應氣候變遷的政策目標是：

1. 強化經濟金融體系因應氣候變遷風險之韌性 [S2]
2. 協助經濟體系順利轉型至永續之綠色經濟 [S2]
 
[source: S2]


❌ NGFS（綠色金融體系網絡）是在哪一年、於哪個城市成立的？
   expected: 2017年12月，在巴黎（第一次「一個星球峰會」中）成立
   actual  : NGFS（綠色金融體系網絡）是在 2017 年，在巴黎成立的。 

根據上述文獻 [S1] (因應氣候變遷策略方案.pdf 第20頁)，NGFS 係於 2017 年 12 月在巴黎舉行之第一次「一個星球峰會(One Planet Summit) 」中成立。

[S1] (因應氣候變遷策略方案.pdf 第20頁)

（Source： [S1]）

In [ ]:
## print the experiments of the final evaluation
from RAG_module.experiment_tracker import show_experiments
show_experiments()

,id,timestamp,chunk_size,chunk_overlap,top_k,reranker_top_k,embedding_model,reranker_model,llm_model,judge_model,recall,notes,answer_quality,citation_validity,faithfulness
0,1,2026-07-20 08:41,256,50,8,8,BAAI/bge-m3,BAAI/bge-reranker-v2-m3,llama-3.1-8b-instant,llama-3.3-70b-versatile,0.89,,NaN,NaN,NaN
1,2,2026-07-20 08:47,256,50,8,8,BAAI/bge-m3,BAAI/bge-reranker-v2-m3,llama-3.1-8b-instant,llama-3.3-70b-versatile,NaN,,0.9,1.0,0.9
2,3,2026-07-20 08:52,256,50,8,8,BAAI/bge-m3,cross-encoder/ms-marco-MiniLM-L-6-v2,llama-3.1-8b-instant,llama-3.3-70b-versatile,0.78,,0.8,1.0,0.8
